# Aberration Simulation GPU Smoke Test

Run this notebook from inside the `aberration-simulation-github` project, or keep it in `notebooks/` as provided. It checks the active backend, runs a small aberration-coefficient grid, saves the results, and plots line profiles for nonzero aberration cases.

Expected GPU backend output:

```text
HAS_CUPY: True
backend module: cupy
```

In [ ]:
from pathlib import Path
import os
import sys

def find_repo_root():
    here = Path.cwd().resolve()
    candidates = [here, here.parent, here.parent.parent]
    for candidate in candidates:
        if (candidate / 'src' / 'aberration_simulation').exists():
            return candidate
    raise RuntimeError('Could not find repo root containing src/aberration_simulation')

ROOT = find_repo_root()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
print('repo root:', ROOT)

## Backend Check

If this prints `HAS_CUPY: False`, the test will still run, but it is using NumPy/SciPy instead of GPU CuPy.

In [ ]:
from aberration_simulation.backend import HAS_CUPY, xp

print('HAS_CUPY:', HAS_CUPY)
print('backend module:', xp.__name__)
print('backend version:', getattr(xp, '__version__', 'n/a'))

if HAS_CUPY:
    print('cuda runtime:', xp.cuda.runtime.runtimeGetVersion())
    device = xp.cuda.Device()
    props = xp.cuda.runtime.getDeviceProperties(device.id)
    print('gpu device:', props['name'].decode())

Uncomment this assertion if you want the notebook to fail unless CuPy/GPU is active.

In [ ]:
# assert HAS_CUPY, 'CuPy is not active. Install CuPy or switch to a GPU runtime.'

## Run Reduced Aberration Grid

In [ ]:
from aberration_simulation.backend import asnumpy
from aberration_simulation.optics import SimulationConfig, build_parameter_grid, run_simulation, save_npz

config = SimulationConfig(
    pix_dim=(64, 64),
    real_dim=(320, 320),
    app=30,
    sigma=1,
)

parameters = build_parameter_grid(
    C1_offset_sequence=[0],
    A3_amp_sequence=[0, 20],
    A2_amp_sequence=[0, 2],
    C1_sequence=[0],
    C3_sequence=[0, 0.3],
    A1_amp_sequence=[0, 20],
    A1_phase_sequence=[0],
    A2_phase_sequence=[0, 60],
    A3_phase_sequence=[0, 90],
)

probe_images, selected = run_simulation(config, parameters)
probe_np = asnumpy(probe_images)

print('backend:', 'cupy' if HAS_CUPY else 'numpy/scipy')
print('parameter combinations:', len(selected))
print('probe image shape:', probe_np.shape)
print('intensity range:', float(probe_np.min()), float(probe_np.max()))

assert probe_np.shape == (64, 64, 64)
assert probe_np.max() > probe_np.min()
assert not xp.isnan(probe_images).any()

output_dir = ROOT / 'outputs'
output_dir.mkdir(exist_ok=True)
save_npz(output_dir / 'notebook_smoke_probe_images.npz', probe_images, selected)
print('saved:', output_dir / 'notebook_smoke_probe_images.npz')

## Select Nonzero Aberration Cases

In [ ]:
from aberration_simulation.line_profiles import choose_nonzero_parameter_indices

indices = choose_nonzero_parameter_indices(selected, limit=4)
selected_profiles = [selected[i] for i in indices]

for image_index, params in zip(indices, selected_profiles):
    print(image_index, params)

## Plot Probe Images

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(indices), figsize=(4 * len(indices), 4))
if len(indices) == 1:
    axes = [axes]

for ax, image_index, params in zip(axes, indices, selected_profiles):
    ax.imshow(probe_np[:, :, image_index], cmap='magma')
    ax.set_title(
        f"A1={params['A1_amp']:g}, A2={params['A2_amp']:g}, "
        f"A3={params['A3_amp']:g}, C3={params['C3']:g}",
        fontsize=9,
    )
    ax.set_axis_off()

fig.tight_layout()

## Extract and Plot Line Profiles

In [ ]:
from aberration_simulation.line_profiles import extract_line_profiles_from_stack

stack = probe_images[:, :, indices]
profiles, coords = extract_line_profiles_from_stack(stack, num_lines=9, radius=24)
profiles_np = asnumpy(profiles)

print('line profiles shape:', profiles_np.shape)
assert profiles_np.shape == (8, 49, len(indices))

In [ ]:
plot_dir = ROOT / 'outputs' / 'plots'
plot_dir.mkdir(parents=True, exist_ok=True)

for local_index, params in enumerate(selected_profiles):
    fig, ax = plt.subplots(figsize=(7, 4))
    for angle_index, angle in enumerate(coords['angles_deg']):
        ax.plot(profiles_np[angle_index, :, local_index], label=f'{angle:.0f} deg')
    ax.set_title(
        f"Line profiles: A1={params['A1_amp']:g}, A2={params['A2_amp']:g}, "
        f"A3={params['A3_amp']:g}, C3={params['C3']:g}",
        fontsize=10,
    )
    ax.set_xlabel('pixel along line')
    ax.set_ylabel('intensity')
    ax.legend(ncol=2, fontsize=8)
    fig.tight_layout()
    path = plot_dir / f'notebook_line_profiles_{local_index:02d}.png'
    fig.savefig(path, dpi=160)
    print('saved:', path)
    plt.show()